# Advanced Claude Platform on AWS

**The second notebook in the Acme Parts Co. series.**

This notebook picks up where the [Getting Started notebook](getting-started-claude-platform-on-aws.ipynb) left off. You've already built the core support bot — messages, vision, conversations, streaming, thinking, and custom tools. Now we go deeper: document management, cost optimization, server-hosted tools, external integrations, and production observability.

**Assumptions**
- You've completed the Getting Started notebook
- Same `.env` credentials (CP-on-AWS API key + workspace ID)
- For the observability chapter: AWS admin credentials with CloudTrail/Cost Explorer access (optional)

**Length**: ~20 minutes reading + running.

**Cost**: under $0.50 end-to-end if you run every cell once.

---

### Chapter map

0. Setup
1. Files API — uploading supplier spec sheets
2. Citations — verifiable answers
3. Prompt caching + token counting
4. Searching the web — product research
5. Running code — shipping math
6. Memory — remembering customer preferences
7. Editing files — fixing the knowledge base
8. MCP connector — plugging in external tool servers
9. Computer use — the future (synthetic demo)
10. Observability — CloudTrail and Cost Explorer
11. Production auth — IdP + IAM Identity Center
12. Wrap-up

## Install dependencies

Runs once per environment. Skip if you've already installed `requirements.txt` in your venv.

In [ ]:
%pip install -q anthropic python-dotenv Pillow

## 0. Setup

Credentials live in a local `.env` file so you don't paste secrets into the notebook. Copy `.env.example` to `.env` and fill in three values:

```env
ANTHROPIC_AWS_API_KEY=aws-external-anthropic-api-key-...
ANTHROPIC_AWS_WORKSPACE_ID=wrkspc_...
AWS_REGION=us-east-2
CLAUDE_MODEL=claude-sonnet-4-6
```

The short-term API key is issued from the CP-on-AWS console and expires every 12 hours. Rotate as needed.

If you'd rather skip the `.env` dance, paste the values into the fallback block below directly.

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import AnthropicAWS

load_dotenv()  # no-op if .env is missing

# ── Fallback: if you don't want a .env, paste values here ──────────────
# KEY       = "aws-external-anthropic-api-key-..."
# WORKSPACE = "wrkspc_..."
# REGION    = "us-east-2"
# MODEL     = "claude-sonnet-4-6"
# ──────────────────────────────────────────────────────────────────────

KEY       = os.environ.get("ANTHROPIC_AWS_API_KEY")
WORKSPACE = os.environ.get("ANTHROPIC_AWS_WORKSPACE_ID")
REGION    = os.environ.get("AWS_REGION", "us-east-2")
MODEL     = os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-6")

missing = [name for name, val in
           [("ANTHROPIC_AWS_API_KEY", KEY), ("ANTHROPIC_AWS_WORKSPACE_ID", WORKSPACE)]
           if not val]
if missing:
    raise RuntimeError(
        f"Missing credentials: {', '.join(missing)}. "
        "Add them to a .env file next to this notebook, or paste into the fallback block above and re-run this cell."
    )

# One client object drives every chapter.
# AnthropicAWS handles the CP-on-AWS endpoint, bearer auth,
# workspace header, and retries for you.
client = AnthropicAWS(
    api_key=KEY,
    aws_region=REGION,
    workspace_id=WORKSPACE,
    timeout=60.0,
    max_retries=3,
)

print(f"Region:    {REGION}")
print(f"Workspace: {WORKSPACE}")
print(f"Model:     {MODEL}")

## Chapter 1 — Files API: uploading supplier spec sheets

Acme's catalog team maintains hundreds of supplier spec sheets — PDF datasheets for bearings, gaskets, motors, and more. Every time a customer asks *"Is the BRG-1050 rated for high-temperature environments?"*, the support bot needs to consult the right spec sheet.

Re-uploading a 15-page PDF on every API call is wasteful. The **Files API** lets you upload a document once, get back a `file_id`, and reference it in as many Messages requests as you like. Upload once, query many times.

**What the Files API gives you:**
- Upload PDFs, images, and text files to Anthropic-managed storage
- Reference them by `file_id` in any Messages call — no base64 re-encoding
- List, inspect, and delete files when they go stale
- Free storage operations — you only pay input tokens when the file is used in a message

> **Beta note**: The Files API requires the beta header `files-api-2025-04-14`. The SDK handles this when you use `client.beta.files` and pass `betas=[\"files-api-2025-04-14\"]` to `client.beta.messages.create`.

### Step 1 — Create a sample spec sheet and upload it

In [ ]:
from pathlib import Path

# ── Create a realistic supplier spec sheet (plain text, simulating a PDF) ──
# In production you'd upload real PDFs from your catalog.
# Here we use a .txt file so the example runs without extra dependencies.

SPEC_DIR = Path("assets")
SPEC_DIR.mkdir(exist_ok=True)

spec_text = """ACME PARTS CO. — SUPPLIER SPECIFICATION SHEET
=================================================

Part Number:    BRG-1050
Description:    Deep-groove ball bearing, single row
Supplier:       Nachi-Fujikoshi Corp.
Category:       Bearings > Ball Bearings > Deep Groove

DIMENSIONS
----------
  Inner diameter (d):   50 mm
  Outer diameter (D):   110 mm
  Width (B):            27 mm
  Weight:               0.98 kg

LOAD RATINGS
------------
  Dynamic (Cr):         62.4 kN
  Static  (C0r):        43.5 kN
  Limiting speed:       6,300 rpm (grease) / 8,000 rpm (oil)

OPERATING CONDITIONS
--------------------
  Temperature range:    -30 °C to +120 °C (standard seals)
                        -30 °C to +200 °C (high-temp variant BRG-1050H)
  Lubrication:          Pre-greased (Mobil Polyrex EM)
  Seal type:            2RS (contact rubber seals both sides)
  Corrosion resistance: Standard (ZZ variant available for washdown environments)

COMPLIANCE
----------
  ISO 492 (tolerance class Normal)
  REACH / RoHS compliant
  Cage material: Pressed steel (standard) or Polyamide PA66 (option)

PACKAGING & MOQ
----------------
  Inner pack:   1 unit / box
  Carton:       20 units
  MOQ:          20 units
  Lead time:    2-4 weeks (domestic stock), 8-12 weeks (factory order)

PRICING (Acme distributor cost)
----------------------------------
  1-19 units:    $42.00 / unit
  20-99 units:   $38.50 / unit
  100-499 units: $34.75 / unit
  500+ units:    $31.00 / unit (annual contract required)

CROSS-REFERENCE
----------------
  SKF:   6210-2RS1
  NSK:   6210DDU
  NTN:   6210LLU
  FAG:   6210-2RSR

NOTES
-----
  - For high-temperature applications above 120 °C, order BRG-1050H.
  - Minimum reorder point recommendation: 40 units (based on 90-day avg demand).
  - Warranty: 24 months from date of manufacture or 12 months from installation.
"""

spec_path = SPEC_DIR / "BRG-1050_spec_sheet.txt"
spec_path.write_text(spec_text)
print(f"Created sample spec sheet: {spec_path}  ({spec_path.stat().st_size:,} bytes)")

In [ ]:
# ── Upload the spec sheet via the Files API ──

uploaded_file = client.beta.files.upload(
    file=(spec_path.name, spec_path.open("rb"), "text/plain"),
)

print(f"file_id:      {uploaded_file.id}")
print(f"filename:     {uploaded_file.filename}")
print(f"size:         {uploaded_file.size_bytes:,} bytes")
print(f"mime_type:    {uploaded_file.mime_type}")
print(f"created_at:   {uploaded_file.created_at}")
print()
print("The file now lives in Anthropic-managed storage.")
print("You can reference it by file_id in any future Messages call — no re-upload needed.")

### Step 2 — Ask Claude about the spec sheet using the file_id

Now the support bot can answer product questions by referencing the uploaded file. No base64 encoding, no re-reading the file from disk — just pass the `file_id`.

In [ ]:
msg = client.beta.messages.create(
    model=MODEL,
    max_tokens=600,
    betas=["files-api-2025-04-14"],
    system=(
        "You are Acme Bot, the technical support assistant for Acme Parts Co., "
        "a mid-market parts distributor. Answer questions using the attached spec sheet. "
        "Be precise — cite exact values from the document."
    ),
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "file",
                    "file_id": uploaded_file.id,
                },
                "title": "BRG-1050 Supplier Spec Sheet",
                "context": "Official spec sheet from our bearing supplier catalog.",
            },
            {
                "type": "text",
                "text": (
                    "A customer is asking: 'We run our line at 150 °C. "
                    "Can we use the BRG-1050, or do we need a different variant? "
                    "Also, what's the price break if we order 200 units?'"
                ),
            },
        ],
    }],
)

for block in msg.content:
    if block.type == "text":
        print("🤖", block.text)
print(f"\n({msg.usage.input_tokens} in / {msg.usage.output_tokens} out tokens)")

### Step 3 — Reuse the same file in a different question

This is the payoff: the file is already stored. A second customer question about the same part costs zero upload time.

In [ ]:
msg2 = client.beta.messages.create(
    model=MODEL,
    max_tokens=400,
    betas=["files-api-2025-04-14"],
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {"type": "file", "file_id": uploaded_file.id},
            },
            {
                "type": "text",
                "text": "What's the SKF cross-reference for this bearing, and what's the MOQ?",
            },
        ],
    }],
)

for block in msg2.content:
    if block.type == "text":
        print("🤖", block.text)
print(f"\n({msg2.usage.input_tokens} in / {msg2.usage.output_tokens} out tokens)")

### Step 4 — List and clean up files

In production, Acme would upload spec sheets when the catalog updates and delete stale ones. Here's how to manage your file inventory.

In [ ]:
# ── List all files in the workspace ──
print("Files in workspace:")
for f in client.beta.files.list().data:
    print(f"  {f.id}  {f.filename:40s}  {f.size_bytes:>10,} bytes  {f.created_at}")

# ── Delete the demo file to keep things tidy ──
client.beta.files.delete(uploaded_file.id)
print(f"\nDeleted {uploaded_file.id} ({uploaded_file.filename})")

### Files API — key takeaways

| | Inline (base64 / text) | Files API |
|---|---|---|
| **Upload cost** | Every request re-sends the full document | Upload once, reference by `file_id` |
| **Supported types** | PDF, images, plain text | PDF, images, plain text (+ more via code execution) |
| **Storage** | None — ephemeral per request | Persistent until you delete (500 MB/file, 500 GB/org) |
| **Best for** | One-off analysis, small files | Catalogs, spec sheets, reference docs queried repeatedly |
| **Billing** | Input tokens per request | Storage ops free; input tokens when used in Messages |

For a parts distributor like Acme, the Files API is the natural way to build a product-knowledge layer: upload the catalog once, let every support query reference the right spec sheet by `file_id`.

> **Platform note**: The Files API is currently available on the native Anthropic API and Claude Platform on AWS. It is not yet supported on Amazon Bedrock or Google Vertex AI.

## Chapter 2 — Citations: verifiable answers from spec sheets

When a support agent tells a customer *"the bearing is rated to 120 °C"*, the customer might ask: *"Where does it say that?"* Citations solve this. Enable `citations` on a document block and Claude's response includes exact quotes with character-level pointers back to the source.

This is especially valuable for Acme: spec sheets are the source of truth, and every claim the bot makes should be traceable to a specific line in the document.

Let's re-upload the spec sheet and ask the same temperature question — but this time with citations enabled.

In [ ]:
# ── Upload the spec sheet again (the previous one was deleted in Step 4) ──
cited_file = client.beta.files.upload(
    file=(spec_path.name, spec_path.open("rb"), "text/plain"),
)
print(f"Uploaded: {cited_file.id}  ({cited_file.filename})")

In [ ]:
# ── Ask a question with citations enabled ──

msg = client.beta.messages.create(
    model=MODEL,
    max_tokens=600,
    betas=["files-api-2025-04-14"],
    system=(
        "You are Acme Bot, the technical support assistant for Acme Parts Co. "
        "Answer questions using the attached spec sheet. Be precise."
    ),
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "file",
                    "file_id": cited_file.id,
                },
                "title": "BRG-1050 Supplier Spec Sheet",
                # ⬇️ This is the only addition — one field enables citations.
                "citations": {"enabled": True},
            },
            {
                "type": "text",
                "text": (
                    "A customer asks: 'What's the max operating temperature for the BRG-1050, "
                    "and what should I order if I need higher temps? Also, what's the warranty?'"
                ),
            },
        ],
    }],
)

# ── Display the response with citations ──
for block in msg.content:
    if block.type != "text":
        continue
    citations = getattr(block, "citations", None) or []
    if citations:
        # This text block has citations — show the claim and its sources
        print(f"📌 \"{block.text}\"")
        for cite in citations:
            cited_text = getattr(cite, "cited_text", "")
            start = getattr(cite, "start_char_index", "?")
            end = getattr(cite, "end_char_index", "?")
            doc_title = getattr(cite, "document_title", "?")
            print(f"   ↳ [{doc_title} chars {start}–{end}]: \"{cited_text}\"")
    else:
        # Plain text (connective tissue between cited claims)
        print(block.text, end="")

print(f"\n\n({msg.usage.input_tokens} in / {msg.usage.output_tokens} out tokens)")

Each `📌` line is a claim Claude made, and the `↳` lines show the exact text it cited from the spec sheet — with character offsets you can use to highlight the source in a UI.

**What changed?** One field on the document block:

```python
\"citations\": {\"enabled\": True}
```

**How the response differs:**
- Without citations: one big `text` block with the answer
- With citations: multiple `text` blocks, some with a `citations` list containing `char_location` objects (for plain text) or `page_location` objects (for PDFs)

**Cost note**: `cited_text` in the response doesn't count toward output tokens, so citations are essentially free on the output side. There's a small input-token overhead for the chunking metadata.

**For PDFs**: if you upload a real PDF instead of a text file, citations include `start_page_number` and `end_page_number` instead of character indices — perfect for linking back to a specific page in the datasheet.

In [ ]:
# ── Clean up the cited file ──
client.beta.files.delete(cited_file.id)
print(f"Deleted {cited_file.id}")

## Chapter 3 — Prompt caching and token counting

There's a cost problem hiding in multi-turn conversations. Every call re-sends the full conversation history — system prompt, every prior user turn, every prior assistant turn. On turn 10 of a support chat, you're paying full input-token price for the same 9 turns the model already processed.

**Prompt caching** fixes this. Add a single `cache_control` parameter at the top level of your request and the API automatically caches the growing conversation prefix. On the next call, everything up to the last turn is read from cache at **10% of the normal input price**.

The simplest approach is **automatic caching** — one parameter, no per-block markers. The cache breakpoint moves forward automatically as the conversation grows.

| | Without caching | With caching (turn 2+) |
|---|---|---|
| System prompt + prior turns | Full input price | 10% (cache read) |
| New turn content | Full input price | Full input price |
| First-call write surcharge | — | 25% extra (one-time) |
| Cache lifetime | — | 5 min (auto-refreshed on use) |

In [ ]:
# ── Same support bot, now with prompt caching ──
#
# IMPORTANT: Prompt caching has a minimum token threshold.
# For Sonnet 4.6: 2,048 tokens. For Sonnet 4.5/Opus 4.1/Opus 4: 1,024 tokens.
# If your prompt is shorter, caching is silently skipped (no error, just no cache).
#
# We load a realistic, detailed system prompt from a file — the kind a production
# support bot would actually have. See assets/acme_system_prompt.txt to review
# or customize it.

SYSTEM_CACHED = Path("assets/acme_system_prompt.txt").read_text()
print(f"System prompt loaded: {len(SYSTEM_CACHED):,} chars")

cached_conversation = []

def cached_turn(user_text: str) -> dict:
    """Send a turn with automatic prompt caching enabled."""
    cached_conversation.append({"role": "user", "content": user_text})
    resp = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=SYSTEM_CACHED,
        messages=cached_conversation,
        # ⬇️ This is the only addition — one line enables automatic caching.
        cache_control={"type": "ephemeral"},
    )
    cached_conversation.append({"role": "assistant", "content": resp.content})
    text = next(b.text for b in resp.content if b.type == "text")
    return {"text": text, "usage": resp.usage}

# Turn 1 — likely below the cache threshold (system + 1 short message)
# Caching may not activate yet — that's expected.
r1 = cached_turn("I need to return a damaged bearing from order PS-48219.")
print("👤 I need to return a damaged bearing from order PS-48219.")
print("🤖", r1["text"])
print(f"   ⮑ input: {r1['usage'].input_tokens}  "
      f"cache_write: {r1['usage'].cache_creation_input_tokens}  "
      f"cache_read: {r1['usage'].cache_read_input_tokens}")
print()

# Turn 2 — system + turn 1 (user + assistant) should cross the threshold
r2 = cached_turn("Will you cover the return shipping?")
print("👤 Will you cover the return shipping?")
print("🤖", r2["text"])
print(f"   ⮑ input: {r2['usage'].input_tokens}  "
      f"cache_write: {r2['usage'].cache_creation_input_tokens}  "
      f"cache_read: {r2['usage'].cache_read_input_tokens}")
print()

# Turn 3 — cache hit: system + turns 1-2 read from cache
r3 = cached_turn("Great, what's the return address?")
print("👤 Great, what's the return address?")
print("🤖", r3["text"])
print(f"   ⮑ input: {r3['usage'].input_tokens}  "
      f"cache_write: {r3['usage'].cache_creation_input_tokens}  "
      f"cache_read: {r3['usage'].cache_read_input_tokens}")

Look at the `cache_read` numbers climbing across turns — that's the system prompt and earlier turns being served from cache instead of reprocessed. The `input` count (uncached tokens) stays small because it's only the new turn content.

**What changed in the code?** Exactly one line:

```python
cache_control={\"type\": \"ephemeral\"},
```

That's it. The SDK handles the rest — the cache breakpoint automatically moves to the end of the conversation on each call.

**Minimum token threshold**: prompt caching requires a minimum number of tokens before it kicks in (1,024 for Sonnet 4.5/Opus 4, 2,048 for Sonnet 4.6, 4,096 for Opus 4.5+). If your prompt is shorter, caching is silently skipped — no error, just `cache_write` and `cache_read` both showing 0. That's why we use a detailed system prompt above: a production support bot would have one anyway, and it needs to be long enough to cross the threshold.

**Expected behavior across turns:**
- **Turn 1**: `cache_write` is high (writing the system prompt + first turn to cache), `cache_read` is 0
- **Turn 2**: `cache_read` climbs (system + turn 1 served from cache), `cache_write` is smaller (only the new content)
- **Turn 3+**: `cache_read` keeps growing as more history is served from cache

**Cost math for Acme**: if the support bot averages 8 turns per session with a ~600-token system prompt, caching saves roughly 70-80% on input tokens across the session. At Acme's volume (~2,000 support chats/day), that adds up fast.

> **Tip**: The cache has a 5-minute TTL, refreshed each time it's used. For support bots where customers might go idle, consider `cache_control={\"type\": \"ephemeral\", \"ttl\": \"1h\"}` (1-hour cache at 2× base input price for writes — still cheaper than re-processing long conversations).

### Estimating cost before you send: token counting

Before Acme commits to a long API call — say, sending a 50-page spec sheet with a multi-turn conversation — the ops team wants to know: *how much will this cost?*

The **Token Count API** (`client.messages.count_tokens(...)`) takes the same parameters as `messages.create` and returns the input token count *without* actually running the model. No output tokens, no charge — just a count you can use for budgeting, rate-limit checks, or deciding whether to truncate context.

In [ ]:
# ── Count tokens before sending ──

token_count = client.messages.count_tokens(
    model=MODEL,
    system=SYSTEM_CACHED,
    messages=cached_conversation,  # the 3-turn conversation from above
)

print(f"Input tokens: {token_count.input_tokens:,}")
print()
print("This is what you'd pay for (input side) if you sent this conversation right now.")
print("Use this to:")
print("  • Budget API spend before committing to a call")
print("  • Decide whether to summarize/truncate older turns")
print("  • Check you're within rate limits")
print("  • Compare cached vs uncached cost projections")

## Chapter 4 — Searching the web: product research

A customer asks: *"Is the BRG-1050 bearing compatible with a Yamaha MT-07?"* Your internal database doesn't know, and you don't want to make something up.

The **web_search** tool is an Anthropic-hosted server tool. You declare it; Anthropic runs the search; Claude gets the results and answers with citations. You never execute anything locally.

Cost note: web_search charges per search (separate from token cost), so cap it with `max_uses`.

In [ ]:
msg = client.messages.create(
    model=MODEL,
    max_tokens=700,
    tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 2}],
    messages=[{"role": "user", "content":
        "In 3 sentences, what's new with the Anthropic Python SDK in 2026? Include at least one source URL."}],
)

for block in msg.content:
    if block.type == "text":
        print("💬", block.text)
    elif block.type == "server_tool_use":
        print("🔎 search:", block.input)
    elif block.type == "web_search_tool_result":
        results = block.content if isinstance(block.content, list) else []
        print(f"📄 {len(results)} results")
        for r in results[:3]:
            print(f"   - {getattr(r, 'title', '?')[:70]}")
            print(f"     {getattr(r, 'url', '?')}")

print(f"\n({msg.usage.input_tokens} in / {msg.usage.output_tokens} out, {msg.usage.server_tool_use})")

## Chapter 5 — Running code: shipping math

A regional ops lead pastes a spreadsheet of package weights and dimensions and asks: *"What's the shipping cost breakdown if zone pricing is $0.45/lb under 20 lb, $0.35/lb between 20 and 100 lb, and $0.28/lb above?"*

The **code_execution** tool gives Claude a sandboxed Python + bash environment. It's another server tool — Anthropic runs the container; you just get the answer.

Good for: data analysis, calculations, small scripts, charts, parsing files the user uploads.

In [ ]:
msg = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    tools=[{"type": "code_execution_20260120", "name": "code_execution"}],
    messages=[{"role": "user", "content":
        "Use code execution to compute shipping cost for these 5 packages (weights in lb): "
        "[8, 22, 45, 110, 260]. Zone pricing: $0.45/lb up to 20 lb, $0.35/lb from 20 to 100 lb, $0.28/lb above 100 lb. "
        "Show each package's cost and the total."}],
)

for block in msg.content:
    t = block.type
    if t == "text":
        print("💬", block.text)
    elif t == "server_tool_use":
        code = block.input.get("code", "") if isinstance(block.input, dict) else ""
        if code:
            print("🧪 code Claude ran:")
            print(code)
            print()
    elif "code_execution" in t and "result" in t:
        print("📤 sandbox output:")
        print(str(getattr(block, 'content', block))[:400])

## Chapter 6 — Memory: remembering customer preferences

Regular customers get annoyed repeating preferences every time. The **memory** tool lets Claude persist notes to a file-like store *you* control. Your app owns the storage; Claude just tells it what to write and read.

Think of it as Claude's notebook that survives across conversations.

In [ ]:
# In production this is a database keyed by customer id. Here, a dict.
MEMORY = {}

def run_memory(cmd: dict) -> str:
    op = cmd.get("command")
    path = cmd.get("path", "")
    if op == "view":
        if path in MEMORY:
            return MEMORY[path]
        matches = sorted(k for k in MEMORY if k.startswith(path))
        return "\n".join(matches) or "(empty)"
    if op == "create":
        MEMORY[path] = cmd.get("file_text", "")
        return f"created {path}"
    if op == "str_replace":
        MEMORY[path] = MEMORY.get(path, "").replace(cmd["old_str"], cmd.get("new_str", ""))
        return f"edited {path}"
    if op == "delete":
        MEMORY.pop(path, None)
        return f"deleted {path}"
    return f"unknown op {op}"

tools = [{"type": "memory_20250818", "name": "memory"}]
messages = [{"role": "user", "content":
    "Customer ACME-042 prefers next-day shipping to their Denver warehouse and hates email notifications. "
    "Please save this in /memories/customers/ACME-042.md, then read it back."}]

for turn_idx in range(5):
    resp = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)
    print(f"--- turn {turn_idx}  stop={resp.stop_reason} ---")
    tool_results = []
    for block in resp.content:
        if block.type == "text":
            print("💬", block.text)
        elif block.type == "tool_use":
            print(f"📒 memory.{block.input.get('command')}  path={block.input.get('path')}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                  "content": run_memory(block.input)})
    messages.append({"role": "assistant", "content": resp.content})
    if resp.stop_reason != "tool_use":
        break
    messages.append({"role": "user", "content": tool_results})

print("\nStored in our app's memory:", MEMORY)

## Chapter 7 — Editing files: fixing the knowledge base

Acme has a returns policy doc in `scratch/returns_policy.md` with a typo. The **text_editor** tool gives Claude a structured way to `view`, `create`, `str_replace`, and `insert` into files — you execute the ops in a sandbox you control.

Below we scope everything to a `scratch/` folder. Any path outside it is rejected by our handler.

In [ ]:
from pathlib import Path

SANDBOX = Path("scratch").resolve()
SANDBOX.mkdir(exist_ok=True)

# Seed the doc with a typo
(SANDBOX / "returns_policy.md").write_text(
    "# Acme Returns Policy\n\n"
    "- Items must be returend within 30 days.\n"
    "- Original packaging required.\n"
    "- Refunds issued to original payment method.\n"
)

def safe_path(p: str) -> Path:
    candidate = (SANDBOX / p.lstrip("/")).resolve()
    if SANDBOX not in candidate.parents and candidate != SANDBOX:
        raise ValueError(f"path escapes sandbox: {p}")
    return candidate

def run_editor(op: dict) -> str:
    cmd = op.get("command")
    path = safe_path(op["path"])
    if cmd == "view":
        if path.is_dir():
            return "\n".join(str(p.relative_to(SANDBOX)) for p in sorted(path.rglob("*")))
        return "\n".join(f"{i+1}: {ln}" for i, ln in enumerate(path.read_text().splitlines()))
    if cmd == "create":
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(op.get("file_text", ""))
        return f"created {path.relative_to(SANDBOX)}"
    if cmd == "str_replace":
        text = path.read_text()
        if op["old_str"] not in text:
            return f"error: old_str not found"
        path.write_text(text.replace(op["old_str"], op.get("new_str", ""), 1))
        return f"edited {path.relative_to(SANDBOX)}"
    if cmd == "insert":
        lines = path.read_text().splitlines()
        lines.insert(int(op.get("insert_line", 0)), op.get("insert_text", op.get("new_str", "")))
        path.write_text("\n".join(lines) + "\n")
        return f"inserted into {path.relative_to(SANDBOX)}"
    return f"unknown command: {cmd}"

tools = [{"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}]
messages = [{"role": "user", "content":
    "There's a typo in /returns_policy.md. View the file, fix any spelling mistakes, then view it again to confirm."}]

for turn_idx in range(6):
    resp = client.messages.create(model=MODEL, max_tokens=700, tools=tools, messages=messages)
    print(f"--- turn {turn_idx}  stop={resp.stop_reason} ---")
    tool_results = []
    for block in resp.content:
        if block.type == "text":
            print("💬", block.text)
        elif block.type == "tool_use":
            print(f"📝 {block.input.get('command')} on {block.input.get('path')}")
            try:
                result = run_editor(block.input); is_error = False
            except Exception as e:
                result = f"error: {e}"; is_error = True
            tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                  "content": result, "is_error": is_error})
    messages.append({"role": "assistant", "content": resp.content})
    if resp.stop_reason != "tool_use":
        break
    messages.append({"role": "user", "content": tool_results})

print("\n--- Final returns_policy.md ---")
print((SANDBOX / "returns_policy.md").read_text())

## Chapter 8 — MCP connector: plugging in external tool servers

So far, every tool has been either Anthropic-hosted (web search, code execution) or implemented in *your* code (order lookup, memory, text editor). But Acme's warehouse team already runs an **inventory service** with a Model Context Protocol (MCP) endpoint — a standardized way to expose tools to AI models over HTTPS.

The **MCP connector** lets Claude call tools on a remote MCP server directly from the Messages API. You don't need to build an MCP client — just point Claude at the server URL and it handles discovery, invocation, and result parsing.

**How it works:**
1. You declare the MCP server in `mcp_servers` (URL + optional auth token)
2. You add an `mcp_toolset` entry in `tools` to control which tools Claude can use
3. Claude discovers the server's tools, calls them as needed, and weaves the results into its answer

The example below shows the pattern against Acme's (hypothetical) inventory MCP server. Replace the URL with your own server to run it live.

> **Beta note**: The MCP connector requires `betas=[\"mcp-client-2025-11-20\"]`. It's available on the native Anthropic API and Claude Platform on AWS, but not yet on Amazon Bedrock or Google Vertex AI.

In [ ]:
# ── MCP Connector: connect Claude to a remote tool server ──
#
# Replace the URL below with your own MCP server.
# For testing, you can use the official MCP reference server:
#   https://example-server.modelcontextprotocol.io/mcp  (requires OAuth)
# Or deploy your own authless server on Cloudflare Workers.
#
# This cell will raise an error if the server is unreachable — that's expected
# if you haven't replaced the placeholder URL yet.

MCP_SERVER_URL = "https://inventory.acme-parts.example/mcp"  # ← Replace with your MCP server
MCP_AUTH_TOKEN = None  # ← Set if your server requires an auth token

mcp_server_def = {
    "type": "url",
    "url": MCP_SERVER_URL,
    "name": "acme-inventory",
}
if MCP_AUTH_TOKEN:
    mcp_server_def["authorization_token"] = MCP_AUTH_TOKEN

try:
    msg = client.beta.messages.create(
        model=MODEL,
        max_tokens=600,
        betas=["mcp-client-2025-11-20"],
        system=SYSTEM,
        mcp_servers=[mcp_server_def],
        tools=[
            # Enable all tools from the MCP server
            {"type": "mcp_toolset", "mcp_server_name": "acme-inventory"},
        ],
        messages=[{
            "role": "user",
            "content": "Check if we have BRG-1050 bearings in stock at the Denver warehouse.",
        }],
    )

    for block in msg.content:
        if block.type == "text":
            print("💬", block.text)
        elif block.type == "mcp_tool_use":
            print(f"🔌 MCP tool call: {block.server_name}.{block.name}({block.input})")
        elif block.type == "mcp_tool_result":
            print(f"📦 MCP result (error={block.is_error}):")
            for c in (block.content or []):
                print(f"   {getattr(c, 'text', c)}")
    print(f"\n({msg.usage.input_tokens} in / {msg.usage.output_tokens} out tokens)")

except Exception as e:
    print(f"⚠️  Could not reach MCP server at {MCP_SERVER_URL}")
    print(f"   Error: {e}")
    print()
    print("This is expected if you haven't replaced the placeholder URL.")
    print("To run this cell:")
    print("  1. Deploy or point to a real MCP server")
    print("  2. Update MCP_SERVER_URL (and MCP_AUTH_TOKEN if needed)")
    print("  3. Re-run this cell")

### Controlling which tools Claude can use

In production, you probably don't want Claude calling every tool on the server. The `mcp_toolset` supports allowlisting and denylisting:

```python
# Allowlist — only enable specific tools
tools=[
    {
        \"type\": \"mcp_toolset\",
        \"mcp_server_name\": \"acme-inventory\",
        \"default_config\": {\"enabled\": False},
        \"configs\": {
            \"check_stock\": {\"enabled\": True},
            \"get_price\": {\"enabled\": True},
        },
    }
]

# Denylist — enable everything except dangerous tools
tools=[
    {
        \"type\": \"mcp_toolset\",
        \"mcp_server_name\": \"acme-inventory\",
        \"configs\": {
            \"delete_inventory\": {\"enabled\": False},
            \"override_price\": {\"enabled\": False},
        },
    }
]
```

You can also connect to **multiple MCP servers** in a single request — for example, Acme's inventory server *and* a shipping-rates server — by adding entries to both `mcp_servers` and `tools`.

> **Why MCP matters for Acme**: instead of re-implementing every internal API as a Claude tool schema, the warehouse team exposes their existing service as an MCP endpoint once, and any AI-powered app in the company can use it. The support bot, the sales dashboard, the ops alerting system — they all connect to the same MCP server.

## Chapter 9 — Computer use (beta, synthetic demo)

Eventually, Acme wants the bot to operate a browser — filing UPS damage claims on behalf of customers. That's what **computer_use** is for.

How it works: Claude never actually sees a screen. It *asks you* for a screenshot, and emits actions (`left_click`, `type`, `scroll`, `key`) based on what you send back. Your code runs the real browser (usually in a Docker container with a virtual display) and feeds the screenshots in.

In this notebook we keep it safe and didactic: we generate a **synthetic desktop image** and feed that to Claude. It still exercises the full round-trip without touching your real screen.

In [ ]:
import io
import base64
from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, display

def make_fake_desktop(w=1024, h=768) -> bytes:
    img = Image.new("RGB", (w, h), (230, 236, 240))
    d = ImageDraw.Draw(img)
    # browser chrome
    d.rectangle([0, 0, w, 60], fill=(60, 60, 70))
    d.rectangle([160, 18, w - 160, 44], fill=(240, 240, 240))
    d.text((175, 24), "https://shipping.acme-parts.example/claims/new", fill=(40, 40, 40))
    # page content
    d.text((60, 100), "File a damage claim", fill=(15, 25, 40))
    d.rectangle([60, 150, 500, 185], fill=(255, 255, 255), outline=(120, 130, 140))
    d.text((68, 158), "Order number", fill=(120, 130, 140))
    d.rectangle([60, 210, 500, 245], fill=(255, 255, 255), outline=(120, 130, 140))
    d.text((68, 218), "Describe the damage", fill=(120, 130, 140))
    d.rectangle([60, 290, 220, 330], fill=(40, 110, 200))
    d.text((110, 302), "Submit", fill=(255, 255, 255))
    buf = io.BytesIO(); img.save(buf, format="PNG"); return buf.getvalue()

png_bytes = make_fake_desktop()
png_b64 = base64.b64encode(png_bytes).decode()
print("This is what Claude will 'see' when it asks for a screenshot:")
display(IPyImage(data=png_bytes))

tools = [{"type": "computer_20251124", "name": "computer",
          "display_width_px": 1024, "display_height_px": 768, "display_number": 1}]
messages = [{"role": "user", "content":
    "Take a screenshot and tell me what page you see and what the next action should be to file a claim."}]

for turn_idx in range(3):
    resp = client.beta.messages.create(
        model="claude-sonnet-4-6", max_tokens=600,
        betas=["computer-use-2025-11-24"], tools=tools, messages=messages,
    )
    print(f"\n--- turn {turn_idx}  stop={resp.stop_reason} ---")
    tool_results = []
    for block in resp.content:
        if block.type == "text":
            print("💬", block.text)
        elif block.type == "tool_use":
            action = block.input.get("action")
            print(f"🖱  action: {action}")
            if action == "screenshot":
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                    "content": [{"type": "image",
                                 "source": {"type": "base64", "media_type": "image/png", "data": png_b64}}]})
            else:
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                      "content": f"(synthetic) performed {action}"})
    messages.append({"role": "assistant", "content": resp.content})
    if resp.stop_reason != "tool_use":
        break
    messages.append({"role": "user", "content": tool_results})

## Chapter 10 — Observability: who's calling Claude?

The support bot is live. Acme's platform team now asks: *"How do we know who's using it, how often, and how much it costs?"*

Because Claude Platform runs on AWS, API calls are logged to **AWS CloudTrail** under the event source `aws-external-anthropic.amazonaws.com` — the same audit infrastructure you already use for every other AWS service.

**What CloudTrail captures for CP-on-AWS:**
- **Control-plane calls** (account status, workspace management) appear as distinct, named management events (e.g. `GetAccountStatus`, `ListWorkspaces`). These are logged by default.
- **Data-plane calls** (inference via `/v1/messages`, Files API) are logged as **data events**, which CloudTrail does **not** capture by default. You must explicitly enable data event logging on your trail to see these. Once enabled, CloudTrail captures the call metadata (who, when, which workspace, which model) but **not** the prompt or completion content.
- **User identity** varies by auth path: IAM profile calls show the real IAM user ARN; short-term and long-term API keys show a bearer-token identity

The cell below queries CloudTrail for recent CP-on-AWS events. This requires IAM permissions (`cloudtrail:LookupEvents`) — if you're running with just a CP-on-AWS API key, the cell will fail gracefully.

> **Note**: CloudTrail queries use standard AWS credentials (IAM / SSO), not the CP-on-AWS API key. You'll need a separate `boto3` session with the right permissions. Events typically take 5-15 minutes to appear after the API call.

In [ ]:
# ── CloudTrail: audit CP-on-AWS API activity ──
#
# The event source for Claude Platform on AWS is:
#   aws-external-anthropic.amazonaws.com
#
# This is derived from the SigV4 signing service name (aws-external-anthropic)
# following the standard AWS convention: <service>.amazonaws.com.
#
# Prerequisites:
#   - boto3 installed (already in requirements.txt via anthropic's deps)
#   - AWS credentials with cloudtrail:LookupEvents permission
#   - If using SSO: aws sso login --profile your-admin-profile
#   - CloudTrail must be enabled in your region (management events are on by default,
#     but data events — including inference calls — must be enabled separately on your trail)

import boto3
from datetime import datetime, timedelta, timezone
import json as _json

# CP-on-AWS event source — this is the string CloudTrail uses.
CP_EVENT_SOURCE = "aws-external-anthropic.amazonaws.com"

# ── Configure your admin session here ──
# Option A: default credentials (env vars, instance profile, etc.)
ct_session = boto3.Session(region_name=REGION)
#
# Option B: SSO profile (uncomment and set your profile)
# ct_session = boto3.Session(profile_name="your-admin-profile", region_name=REGION)

try:
    ct = ct_session.client("cloudtrail")
    now = datetime.now(timezone.utc)

    # Look up events from the last 24 hours.
    # CloudTrail's LookupEvents API is paginated; we pull up to 20 events.
    resp = ct.lookup_events(
        LookupAttributes=[
            {"AttributeKey": "EventSource", "AttributeValue": CP_EVENT_SOURCE},
        ],
        StartTime=now - timedelta(hours=24),
        EndTime=now,
        MaxResults=20,
    )

    events = resp.get("Events", [])
    if events:
        print(f"Found {len(events)} CP-on-AWS event(s) in the last 24 hours:\n")
        print(f"  {'TIME':<22} {'EVENT NAME':<30} {'MGMT/DATA':<12} {'IDENTITY'}")
        print(f"  {'-'*90}")
        for evt in events:
            # Parse the full CloudTrail event body for details
            ct_body = {}
            raw = evt.get("CloudTrailEvent", "")
            if isinstance(raw, str) and raw:
                try:
                    ct_body = _json.loads(raw)
                except _json.JSONDecodeError:
                    pass

            etime = evt['EventTime'].strftime('%Y-%m-%d %H:%M:%S')
            ename = evt.get('EventName', '?')

            # Management vs data event
            mgmt = ct_body.get('managementEvent')
            mgmt_str = 'management' if mgmt is True else 'data' if mgmt is False else '?'

            # User identity — shows IAM user for profile auth,
            # bearer-token identity for API key auth
            ui = ct_body.get('userIdentity', {})
            id_type = ui.get('type', '?')
            id_arn = ui.get('arn', '') or ui.get('userName', '') or ui.get('principalId', '')
            # Truncate long ARNs for display
            id_short = id_arn if len(id_arn) <= 40 else '...' + id_arn[-37:]

            print(f"  {etime:<22} {ename:<30} {mgmt_str:<12} {id_type}: {id_short}")

        # Quick summary of distinct event names
        names = sorted({e.get('EventName', '') for e in events if e.get('EventName')})
        print(f"\nDistinct event names: {', '.join(names)}")
    else:
        print(f"No events from {CP_EVENT_SOURCE} in the last 24 hours.")
        print("This is normal if:")
        print("  - You just started using CP-on-AWS (events take 5-15 min to appear)")
        print("  - CloudTrail management events aren't enabled in this region")
        print("  - Data events aren't enabled on your trail (required for inference calls like CreateMessage)")
        print("  - No CP-on-AWS calls were made from this account recently")

except Exception as e:
    print(f"⚠️  Could not query CloudTrail: {e}")
    print()
    print("This is expected if you're running with just a CP-on-AWS API key.")
    print("To run this cell, use AWS credentials with cloudtrail:LookupEvents permission.")

### What's in a CloudTrail event?

Each event from `aws-external-anthropic.amazonaws.com` includes:

| Field | What it tells you |
|---|---|
| `eventName` | The API action (e.g. `CreateMessage`, `GetAccountStatus`, `ListWorkspaces`) |
| `eventTime` | When the call happened (UTC) |
| `managementEvent` | `true` for control-plane calls (logged by default); `false` for data-plane calls like `CreateMessage` (requires data events to be enabled on your trail) |
| `userIdentity.type` | `IAMUser` or `AssumedRole` for profile auth; bearer-token identity for API keys |
| `userIdentity.arn` | The IAM principal that made the call |
| `requestParameters` | Call metadata (model, workspace) — **not** the prompt or completion content |
| `sourceIPAddress` | Where the call came from |

**Privacy note**: CloudTrail captures call metadata only. Prompt content and model responses do **not** appear in CloudTrail event bodies — verified by our internal testing. This means you get full audit coverage without sensitive data leaking into your log pipeline.

### Cost tracking with AWS Cost Explorer

CP-on-AWS usage appears on your AWS bill under **Claude Platform**. This means your existing AWS billing workflows, budgets, and alerts work out of the box.

**How billing flows:**
- Token usage (input + output) is metered per API call
- Charges appear under **Claude Platform** on your consolidated bill
- Spend draws down from your **EDP (Enterprise Discount Program)** or **PPA (Private Pricing Agreement)** commitment if you have one
- You can tag IAM principals with attributes like `team` or `cost-center`, activate those tags in Billing and Cost Management, and the resulting cost data flows into Cost Explorer and the detailed Cost and Usage Report (CUR)

**Querying cost programmatically:**

```python
import boto3

# Cost Explorer API is always in us-east-1, regardless of your workload region.
# Requires ce:GetCostAndUsage permission.
ce = boto3.client(\"ce\", region_name=\"us-east-1\")

resp = ce.get_cost_and_usage(
    TimePeriod={\"Start\": \"2026-05-01\", \"End\": \"2026-05-05\"},
    Granularity=\"DAILY\",
    Metrics=[\"UnblendedCost\"],
    Filter={
        # CP-on-AWS charges appear under the service name \"Claude Platform\".
        \"Dimensions\": {
            \"Key\": \"SERVICE\",
            \"Values\": [\"Claude Platform\"],
        }
    },
)

for day in resp[\"ResultsByTime\"]:
    cost = day[\"Total\"][\"UnblendedCost\"][\"Amount\"]
    print(f\"  {day['TimePeriod']['Start']}  ${float(cost):.2f}\")

# To break down by cost-allocation tag (e.g. team):
# Add GroupBy=[{\"Type\": \"TAG\", \"Key\": \"team\"}] to the call above.
# Tags must be activated in Billing → Cost allocation tags first.
```

This is shown as a code snippet rather than a runnable cell because Cost Explorer requires a separate IAM role and billing data takes 24-48 hours to appear. Your finance or platform team can adapt this to their dashboards.

**The key takeaway**: CP-on-AWS gives you the same billing observability as any other AWS service — Cost Explorer, CUR, Budgets, and cost-allocation tags all work. No separate Anthropic invoice to reconcile.

## Chapter 11 — Production auth: IdP + IAM Identity Center

The Getting Started notebook uses a short-term API key from the CP-on-AWS console. That's fine for development, but in production Acme needs:
- No manual key rotation
- Per-user identity for audit trails (CloudTrail logs *who* did what)
- Centralized access control via Identity Center permission sets

The solution: federate your IdP (Auth0, Okta, Entra ID, etc.) into **AWS IAM Identity Center**, then use temporary STS credentials instead of API keys.

**The flow:**
```
Auth0 (SAML/OIDC) → AWS IAM Identity Center → temporary STS creds → CP-on-AWS
```

**Prerequisites:**
1. Your IdP configured as external IdP in AWS IAM Identity Center
2. A permission set that grants access to the Claude Platform workspace
3. User has run: `aws sso login --profile your-sso-profile`

**`~/.aws/config` entry:**
```ini
[profile your-sso-profile]
sso_start_url  = https://your-org.awsapps.com/start
sso_region     = us-east-2
sso_account_id = 123456789012
sso_role_name  = ClaudePlatformAccess
region         = us-east-2
```

**Client initialization with SSO credentials:**
```python
import boto3
from anthropic import AnthropicAWS

session = boto3.Session(profile_name=\"your-sso-profile\")
creds   = session.get_credentials().get_frozen_credentials()

client = AnthropicAWS(
    aws_region=\"us-east-2\",
    workspace_id=\"wrkspc_...\",
    aws_access_key=creds.access_key,
    aws_secret_key=creds.secret_key,
    aws_session_token=creds.token,
)
```

From here, every `client.messages.create`, `client.beta.files.upload`, etc. works identically — the only difference is how the client authenticates.

| | API key (Getting Started) | IdP + IAM Identity Center |
|---|---|---|
| **Auth source** | Short-lived bearer token from CP console | Temporary STS creds from SSO session |
| **Rotation** | Manual (12h expiry, paste new key) | Automatic (boto3 refreshes from SSO cache) |
| **User identity** | Workspace-level | Per-user via IdP → Identity Center mapping |
| **CloudTrail** | Bearer-token identity | Real IAM user ARN |
| **Best for** | Dev/testing, notebooks | Production apps, audit trails, per-user access control |

## Chapter 12 — Wrap-up

Combined with the Getting Started notebook, you've now covered the full CP-on-AWS surface:

| Chapter | Feature | API surface |
|---|---|---|
| 1 | Files API | `client.beta.files.upload`, `file_id` document blocks |
| 2 | Citations | `citations: {enabled: true}` on document blocks |
| 3 | Prompt caching + token counting | `cache_control`, `count_tokens` |
| 4 | Web search | `web_search_20260209` server tool |
| 5 | Code execution | `code_execution_20260120` server tool |
| 6 | Memory | `memory_20250818` client tool |
| 7 | Text editor | `text_editor_20250728` client tool |
| 8 | MCP connector | `mcp_servers` + `mcp_toolset` via `client.beta.messages` |
| 9 | Computer use (beta) | `computer_20251124` via `client.beta.messages` |
| 10 | Observability | CloudTrail `lookup_events`, Cost Explorer |
| 11 | Production auth | IdP → IAM Identity Center → STS creds |

### Where to go next

- **Production auth**: federate through an IdP into IAM Identity Center for per-user credentials (Chapter 11)
- **Observability**: CP-on-AWS activity is recorded in AWS CloudTrail under `aws-external-anthropic.amazonaws.com`; use Cost Explorer to track spend by workspace, model, or cost-allocation tag
- **Cost optimization**: combine prompt caching with token counting to budget API spend; use `max_uses` on server tools
- **Deeper tools**: use custom content documents for fine-grained citation control; connect multiple MCP servers for richer agentic workflows
- **Safety**: for computer_use in production, run the browser in a locked-down container; always keep a human-in-the-loop for destructive actions

Happy building.